# Day 21：从 HTML 提取结构化数据

目标：用 Beautiful Soup 解析网页结构并保存表格。先在本地示例上理解选择器；本课无需网络，不会访问或采集外部网站。安装依赖名称为 `beautifulsoup4`，导入名为 `bs4`。

运行前请阅读[环境与运行说明](../docs/setup.md)。本课 `.py` 是教学源文件，配套 Markdown 和 Notebook 自动同步。图形保存到 `outputs/`，设置 `COURSE_SHOW_PLOTS=1` 可显示窗口。


[Python 源文件](Day%2021_HTML_Data_Collection.py) · [Notebook](Day%2021_HTML_Data_Collection.ipynb) · [完整课程目录](../docs/curriculum.md)


In [ ]:
from pathlib import Path
import sys

# 脚本从文件位置定位仓库；Notebook 从当前工作目录向上查找。
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for candidate in (base, *base.parents):
    if (candidate / "Code" / "course_utils.py").is_file():
        code_dir = str(candidate / "Code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        break
else:
    raise FileNotFoundError("找不到课程仓库，请从仓库根目录或 Code 目录启动 Notebook。")
from course_utils import DATA, OUTPUT, finish_plot


## 解析字段与检查缺失值

选择器定位记录和字段；缺失字段要明确跳过或报错，不能把异常文本静默转成零。对真实请求应设置超时、检查状态码、明确编码，并确认来源允许的数据使用方式。


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
html = """<table><tr class="sample"><td class="hours">2.5</td><td class="score">21</td></tr>
<tr class="sample"><td class="hours">5.0</td><td class="score">47</td></tr></table>"""
soup = BeautifulSoup(html, "html.parser")
records = []
for row in soup.select("tr.sample"):
    hours, score = row.select_one(".hours"), row.select_one(".score")
    if hours is None or score is None:
        raise ValueError("Missing required hours/score field")
    records.append({"Hours": float(hours.get_text(strip=True)), "Scores": float(score.get_text(strip=True))})
if not records:
    raise ValueError("没有找到样本行，请检查 HTML 结构和选择器")
frame = pd.DataFrame(records).drop_duplicates()
OUTPUT.mkdir(parents=True, exist_ok=True)
frame.to_csv(OUTPUT / "day21_samples.csv", index=False)
print(frame)


## 练习与检查

增加缺失字段、重复行和非数字文本，观察报错；决定哪些是可恢复坏记录。为采集数据补来源、时间和字段单位，不把示例数据与真实数据混合。
